# EXP-OF-002: attention versus mean+max on the complete sharded bank

Preparation only; this follow-up requires completed attention heads and all 4,407 merged studies. Native sharded provenance, all seven cache pins, global selection and both source heads are verified before fitting. Requires an independently sealed full attention-source bank SHA256, binding scored and unscored training rows. Both attention checkpoints replay weak-holdout/Gold probabilities against their original CSV with declared FP32 device tolerances. A placeholder with no real bank pin refuses execution before any data resolution or fitting. Only head_kind and experiment change; 12 epochs, split, seed and optimizer stay fixed. Same seed does not guarantee identical minibatch order because initialization consumes RNG differently. No MRI/encoder/weight assets mounted; the existing bank and small private metadata are sufficient. Weak AUC measures teacher agreement; Gold58 is diagnostic and excluded from gradients/checkpoint selection. The 1,500-second Python-stage cap reserves 300 seconds for validation and 1,200 for the unchanged head fit; pre-Python Kaggle mounting is outside this timer. Prediction/UID outputs stay private.

In [ ]:
import time
BOOTSTRAP_MONOTONIC=time.monotonic()
print('{"event":"bootstrap_start","diagnostic_schema":1}',flush=True)
import json,os,time,traceback
BOOTSTRAP_DIR='/kaggle/working'
BOOTSTRAP_STAGE='notebook_start'
os.makedirs(BOOTSTRAP_DIR,exist_ok=True)
def bootstrap_checkpoint(stage):
    global BOOTSTRAP_STAGE
    BOOTSTRAP_STAGE=stage
    event={'diagnostic_schema':1,'event':'bootstrap_checkpoint','stage':stage,'unix_seconds':time.time()}
    with open(os.path.join(BOOTSTRAP_DIR,'bootstrap_progress.json'),'w') as stream:
        json.dump(event,stream,indent=2)
    with open(os.path.join(BOOTSTRAP_DIR,'bootstrap_events.jsonl'),'a') as stream:
        stream.write(json.dumps(event)+'\n')
    print(json.dumps(event),flush=True)
def bootstrap_save_failure(error):
    event={'diagnostic_schema':1,'event':'bootstrap_failure','stage':BOOTSTRAP_STAGE,
           'error_type':type(error).__name__,'error':str(error),'traceback':traceback.format_exc(),
           'unix_seconds':time.time()}
    try:
        with open(os.path.join(BOOTSTRAP_DIR,'bootstrap_failure.json'),'w') as stream:
            json.dump(event,stream,indent=2)
    except BaseException as save_error:
        print(json.dumps({'event':'bootstrap_failure_save_error','error_type':type(save_error).__name__}),flush=True)
    print(json.dumps(event),flush=True)
bootstrap_checkpoint('notebook_start')
with open(os.path.join(BOOTSTRAP_DIR,'bootstrap_start.json'),'w') as stream:
    json.dump({'diagnostic_schema':1,'event':'bootstrap_start','unix_seconds':time.time()},stream,indent=2)


In [ ]:
COMPRESSED_PAYLOAD='...'
SETTINGS={'config': {'experiment': 'orthofoundation_frozen_meanmax_v1', 'encoder': 'dinov3_vitl16', 'extraction_variant': 'author_no_rope', 'feature_dim': 1024, 'checkpoint_sha256': '385a775822107b68eaa486336feb982e1ce7bd6d4e8c03ceb482a0bf546f2ff9', 'checkpoint_bytes': 1213056638, 'resolution': 224, 'slices_per_slot': 4, 'fov': 0.92, 'image_batch_size': 16, 'workers': 2, 'pilot_studies': 16, 'max_extract_seconds': 9000, 'head_kind': 'meanmax', 'head_dim': 256, 'head_dropout': 0.2, 'head_epochs': 12, 'head_batch_size': 64, 'head_lr': 0.001, 'head_weight_decay': 0.01, 'weak_holdout_fold': 0, 'seed': 42}, 'source_sha256': {'orthofoundation/__init__.py': 'f3df43207f5d18497b4f698aa91c746bf81dcd4edb74b81784136acd8aa52cf1', 'orthofoundation/cache.py': '700767576eb7b6d9dfa22f77bdf2088411ba4bdf6036c80db58bf608c96f7485', 'orthofoundation/data.py': 'f3b63ab922f0452c065126dc2ef1c87442a0b988dff29d25c201666a8a5310aa', 'orthofoundation/encoder.py': '86defa5f943bfc161603c7b377bbf08a9d5c2a270d01b1df5c039ab59cff7ecb', 'orthofoundation/heads.py': 'c814b1bcbf1ac1322b717c1f1e6ba7b9173357e956e9104b2c77d0351a833f1f', 'orthofoundation/run.py': '740c2e2599516b1876e7d7226bb0f11c51205cbdaec0b6eebf44dc740c659e10', 'orthofoundation/train.py': '928121a030aabf4fc58594da967ad98ff43a7ce915474d69a909ab09652a7dc1', 'sharded_frozen_extract.py': 'ffc2d904e25f7bdbd506f8c6cd58607484ffacc0305a9665fd538778ca122441', 'compare_frozen_heads.py': '277843a391562237589164780925445a5d47483f275f86b0c2d8f40e5cbdd211', 'compare_sharded_heads.py': '5ea847aedf70ae4842845483808b69956ca28266fb4af84ca580474450a1cf98'}, 'source_feature_identity': '83d0eff1e0af84b242a06d1923e1e3408d27259289faa2543f2b96465359ac89', 'global_manifest_sha256': 'b2a87794d768ca3f6ee3968345f840f1028cff004708a2b3e0cfdfcfb2c15e60', 'assets_manifest_sha256': '6feeb03c976483c2b91cca667f4fdbefa998d3620a93eb6006de22a13cbdeb4f', 'metadata_dataset': 'alanchoo/rsna-knee-ortho-global-metadata-v1', 'previous_kernel': 'alanchoo/rsna-knee-ortho-dataset-merge-heads', 'device': 'cpu', 'encoder_forward_calls': 0, 'max_seconds': 1500, 'head_fit_budget_seconds': 1200, 'validation_reserve_seconds': 300, 'source_bank_required_complete': True, 'source_bank_ready_verified_during_preparation': False, 'source_bank_sha256': None, 'comparison_launch_allowed': False, 'source_bank_pin_origin': 'Root must seal full completed immutable attention output before building executable comparison; builder never measures expected pin from current bank'}
try:
    bootstrap_checkpoint('imports')
    from pathlib import Path
    import base64,hashlib,json,os,re,shutil,subprocess,sys,tarfile,time,zlib
    bootstrap_checkpoint('sealed_source_bank_pin')
    if not SETTINGS['comparison_launch_allowed'] or not isinstance(SETTINGS['source_bank_sha256'],str) or not re.fullmatch(r'[0-9a-f]{64}',SETTINGS['source_bank_sha256']):
        raise RuntimeError('Placeholder only: root must seal completed immutable attention full-bank SHA256 and rebuild before comparison')
    CODE=Path('/kaggle/working/sharded_compare_source'); CODE.mkdir(exist_ok=True)
    bootstrap_checkpoint('vendoring')
    for name,encoded in json.loads(zlib.decompress(base64.b64decode(COMPRESSED_PAYLOAD))).items():
        path=CODE/name; path.parent.mkdir(exist_ok=True); path.write_bytes(base64.b64decode(encoded))
        if hashlib.sha256(path.read_bytes()).hexdigest()!=SETTINGS['source_sha256'][name]:
            raise RuntimeError('Vendored comparison source SHA differs: '+name)
    (CODE/'config.json').write_text(json.dumps(SETTINGS['config'],indent=2))
    INPUT=Path('/kaggle/input')
    def find_input(identifier,marker,kind='datasets'):
        owner,slug=identifier.split('/')
        candidates=[INPUT/kind/owner/slug,INPUT/slug,INPUT/'kernels'/owner/slug,INPUT/owner/slug]
        found=[path for path in candidates if (path/marker).is_file()]
        if not found:
            for pattern in [f'*/{slug}',f'*/*/{slug}',f'*/*/*/{slug}']:
                found.extend(path for path in INPUT.glob(pattern) if (path/marker).is_file())
        found=list(dict.fromkeys(found))
        if len(found)!=1:raise RuntimeError(f'Missing/ambiguous private comparison input {identifier}: {found}')
        return found[0]
    bootstrap_checkpoint('global_metadata_resolution')
    metadata=find_input(SETTINGS['metadata_dataset'],'global_metadata_manifest.json')
    manifest_file=metadata/'global_metadata_manifest.json'
    if hashlib.sha256(manifest_file.read_bytes()).hexdigest()!=SETTINGS['global_manifest_sha256']:
        raise RuntimeError('Global metadata manifest differs from prepared comparison pin')
    manifest=json.loads(manifest_file.read_text()); archive=metadata/manifest['archive']['filename']
    if archive.stat().st_size!=manifest['archive']['bytes'] or hashlib.sha256(archive.read_bytes()).hexdigest()!=manifest['archive']['sha256']:
        raise RuntimeError('Small global metadata archive bytes/SHA differ')
    DATA=Path('/kaggle/working/global_metadata'); DATA.mkdir(exist_ok=True)
    bootstrap_checkpoint('global_metadata_unpack')
    with tarfile.open(archive,'r:gz') as tar:
        members=tar.getmembers(); files=[]
        for member in members:
            path=Path(member.name)
            if path.is_absolute() or '..' in path.parts or member.issym() or member.islnk() or not (member.isfile() or member.isdir()):
                raise RuntimeError('Unsafe global metadata archive member')
            if member.isfile():files.append(member.name)
        if len(files)!=len(set(files)) or set(files)!={row['path'] for row in manifest['members']}:
            raise RuntimeError('Global metadata archive members differ')
        tar.extractall(DATA,filter='data')
    shutil.copyfile(manifest_file,DATA/'global_metadata_manifest.json')
    bootstrap_checkpoint('completed_source_bank_resolution')
    previous=find_input(SETTINGS['previous_kernel'],'run/training_report.json','notebooks')/'run'
    env=dict(os.environ); env['PYTHONPATH']=str(CODE)+os.pathsep+env.get('PYTHONPATH','')
    env['OMP_NUM_THREADS']='1'; env['OPENBLAS_NUM_THREADS']='1'
    env['RSNA_SHARDED_COMPARE_BOOTSTRAP_START']=str(BOOTSTRAP_MONOTONIC)
    if SETTINGS['device']=='cuda':
        bootstrap_checkpoint('gpu_selection')
        import torch
        if not torch.cuda.is_available():raise RuntimeError('CUDA unavailable for explicitly selected T4 comparison')
        free=[torch.cuda.mem_get_info(index)[0] for index in range(torch.cuda.device_count())]
        env['CUDA_VISIBLE_DEVICES']=str(max(range(len(free)),key=lambda index:free[index]))
    cmd=[sys.executable,str(CODE/'compare_sharded_heads.py'),'--previous-run-dir',str(previous),'--global-root',str(DATA),
         '--global-manifest-sha256',SETTINGS['global_manifest_sha256'],'--assets-manifest-sha256',SETTINGS['assets_manifest_sha256'],
         '--expected-feature-identity',SETTINGS['source_feature_identity'],'--output-dir','/kaggle/working/run',
         '--source-bank-sha256',SETTINGS['source_bank_sha256'],
         '--config',str(CODE/'config.json'),'--device',SETTINGS['device'],'--max-seconds',str(SETTINGS['max_seconds'])]
    bootstrap_checkpoint('launch')
    remaining=SETTINGS['max_seconds']-(time.monotonic()-BOOTSTRAP_MONOTONIC)
    if remaining<=0:raise RuntimeError('Comparison Python-stage budget exhausted before child')
    subprocess.run(cmd,env=env,check=True,timeout=remaining)
    bootstrap_checkpoint('report')
    report=json.loads(Path('/kaggle/working/run/training_report.json').read_text())
    if not report['head_training_complete'] or report['encoder_forward_calls']!=0:
        raise RuntimeError('Fixed head-only sharded comparison incomplete')
    print(json.dumps({'training_complete':True,'baseline_attention':report['baseline_attention'],
     'meanmax_weak_holdout_metrics':report['weak_holdout_metrics'],'meanmax_gold_metrics':report['gold_metrics'],
     'macro_auc_delta':report['macro_auc_delta_meanmax_minus_attention'],
     'per_label_auc_delta':report['per_label_auc_delta_meanmax_minus_attention'],'total_seconds':report['total_seconds']},indent=2),flush=True)
    bootstrap_checkpoint('complete')
except BaseException as bootstrap_error:
    bootstrap_save_failure(bootstrap_error)
    raise
